# Prompt Testing Notebook

Fast prompt iteration using saved LLM debug JSONs — no retrieval overhead.

**Goal:** Find a prompt that scores synonym-swap cases >= 0.95 while keeping same-author clean docs below 0.95.

**Key docs:**
- `doc00005` — plagiarised, obfuscation=low (synonym swap), correct source scores 0.850 with current prompt
- `doc00007` — plagiarised, obfuscation=low (synonym swap), correct source scores 0.850 with current prompt  
- `doc00002` — clean (same-author), scores 0.850 at threshold=0.85 → FP risk

**Target:** doc5/7 correct source >= 0.95, doc2 candidate < 0.95

In [ ]:
import json
import time
from pathlib import Path
from ollama import chat

OLLAMA_MODEL = "gemma4:26b"
DEBUG_DIR = Path("../pipeline_results/llm_debug")

def load_pairs(doc_id: str, source_doc_id: str) -> list[dict]:
    path = DEBUG_DIR / doc_id / f"{source_doc_id}.json"
    data = json.load(open(path))
    return data["pairs"]

def test_prompt(prompt_fn, pairs: list[dict], label: str = "") -> dict:
    pairs_text = "\n\n".join([
        f"[Pair {i+1}]\n"
        f"SUSPICIOUS: {p['suspicious_text'][:600]}\n"
        f"SOURCE CANDIDATE: {p['source_text'][:600]}"
        for i, p in enumerate(pairs)
    ])
    prompt = prompt_fn(len(pairs), pairs_text)
    t0 = time.time()
    response = chat(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0, "top_p": 0.95, "top_k": 64, "seed": 42},
        think=False,
    )
    elapsed = round(time.time() - t0, 1)
    raw = response.message.content
    import re
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    data = json.loads(match.group()) if match else {}
    score = float(data.get("score", 0.0))
    reasoning = data.get("reasoning", "")[:200]
    print(f"  [{label}] score={score:.3f}  t={elapsed}s")
    print(f"  reasoning: {reasoning}")
    return {"score": score, "reasoning": reasoning, "elapsed": elapsed}

print("Loaded. Model:", OLLAMA_MODEL)

## Load test pairs

In [ ]:
# Doc 5 — correct source (should score >= 0.95)
doc5_correct = load_pairs("part1__suspicious-document00005.txt", "part1__source-document00178.txt")

# Doc 5 — wrong source (should score < 0.50)
doc5_wrong = load_pairs("part1__suspicious-document00005.txt", "part1__source-document00422.txt")

print(f"Doc5 correct pairs: {len(doc5_correct)}")
print(f"Doc5 wrong pairs:   {len(doc5_wrong)}")

In [ ]:
# Doc 7 pairs — need to run pipeline with --debug-llm first if not present
doc7_dir = DEBUG_DIR / "part1__suspicious-document00007.txt"
if doc7_dir.exists():
    doc7_files = list(doc7_dir.glob("*.json"))
    print("Doc7 debug files:", [f.name for f in doc7_files])
    doc7_correct = load_pairs("part1__suspicious-document00007.txt", "part13__source-document06022.txt")
    print(f"Doc7 correct pairs: {len(doc7_correct)}")
else:
    print("Doc7 debug files not found — run pipeline with --debug-llm on doc00007 first")

## Current prompt (baseline)

In [ ]:
def prompt_baseline(n_pairs, pairs_text):
    return (
        f"You are a strict plagiarism detection expert.\n"
        f"Below are {n_pairs} text pair(s). Each pair shows a chunk from a SUSPICIOUS document "
        f"alongside a chunk from a CANDIDATE SOURCE document.\n\n"
        f"{pairs_text}\n\n"
        f"Your task: determine whether the suspicious text was directly copied or closely paraphrased "
        f"from this specific source document.\n\n"
        f"IMPORTANT RULES:\n"
        f"- Score HIGH (>= 0.95) ONLY if multiple pairs show verbatim copying, near-verbatim text, "
        f"or sentence-level paraphrase where unique phrases, names, or sequences are shared.\n"
        f"- Score LOW (< 0.50) if the texts merely discuss the same topic, share common knowledge, "
        f"or use similar vocabulary without specific shared content.\n"
        f"- Topical similarity alone is NOT plagiarism. The suspicious text must reuse specific "
        f"sentences, phrases, or structure from THIS source.\n"
        f"- If fewer than 3 pairs show strong textual overlap, score below 0.50.\n\n"
        f"Respond with ONLY a JSON object — no markdown, no explanation — with keys: "
        f"score (float 0-1), is_likely_source (bool), reasoning (string)."
    )

print("=== DOC5 CORRECT SOURCE ===")
test_prompt(prompt_baseline, doc5_correct, "baseline")
print("\n=== DOC5 WRONG SOURCE ===")
test_prompt(prompt_baseline, doc5_wrong, "baseline")

## Prompt variant 1 — explicit synonym-swap awareness

In [ ]:
def prompt_v1(n_pairs, pairs_text):
    return (
        f"You are a strict plagiarism detection expert.\n"
        f"Below are {n_pairs} text pair(s). Each pair shows a chunk from a SUSPICIOUS document "
        f"alongside a chunk from a CANDIDATE SOURCE document.\n\n"
        f"{pairs_text}\n\n"
        f"Your task: determine whether the suspicious text was copied from this source document, "
        f"including cases where words have been replaced with synonyms (automated obfuscation).\n\n"
        f"IMPORTANT RULES:\n"
        f"- Score HIGH (>= 0.95) if multiple pairs show: verbatim copying, near-verbatim text, "
        f"OR the same sentence structure with synonyms substituted for content words.\n"
        f"- Synonym substitution evidence: same sentence length, same punctuation pattern, "
        f"same named entities (proper nouns survive synonym swap), same narrative sequence.\n"
        f"- Score LOW (< 0.50) if the texts merely share a topic without matching sentence structure "
        f"or named entities.\n"
        f"- Same-author text reuse is NOT plagiarism — only confirm if specific sentences were copied.\n"
        f"- If fewer than 3 pairs show strong overlap (verbatim or structural), score below 0.50.\n\n"
        f"Respond with ONLY a JSON object — no markdown, no explanation — with keys: "
        f"score (float 0-1), is_likely_source (bool), reasoning (string)."
    )

print("=== DOC5 CORRECT SOURCE ===")
test_prompt(prompt_v1, doc5_correct, "v1")
print("\n=== DOC5 WRONG SOURCE ===")
test_prompt(prompt_v1, doc5_wrong, "v1")

## Prompt variants — rubric-based scoring bands

In [ ]:
def prompt_v2(n_pairs, pairs_text):
    """Explicit 'obfuscated = confirmed' rule."""
    return (
        f"You are a strict plagiarism detection expert.\n"
        f"Below are {n_pairs} text pair(s). Each pair shows a chunk from a SUSPICIOUS document "
        f"alongside a chunk from a CANDIDATE SOURCE document.\n\n"
        f"{pairs_text}\n\n"
        f"Your task: determine whether the suspicious text was copied from this source document.\n\n"
        f"IMPORTANT RULES:\n"
        f"- Score HIGH (>= 0.95) if multiple pairs show verbatim copying, near-verbatim text, "
        f"sentence-level paraphrase with shared unique phrases, OR sentence structure preserved "
        f"with content words replaced by synonyms (automated obfuscation).\n"
        f"- If you determine the text is a 'corrupted', 'obfuscated', or 'scrambled' version of "
        f"the source, that IS plagiarism — score >= 0.95.\n"
        f"- Score LOW (< 0.50) ONLY if the texts share a topic but have completely different "
        f"sentence structures and no matching named entities.\n"
        f"- Same-author text reuse across different volumes/editions is NOT plagiarism — score 0.25.\n"
        f"- If fewer than 3 pairs show structural or lexical overlap, score below 0.50.\n\n"
        f"Respond with ONLY a JSON object — no markdown, no explanation — with keys: "
        f"score (float 0-1), is_likely_source (bool), reasoning (string)."
    )

print("=== DOC5 CORRECT SOURCE ===")
test_prompt(prompt_v2, doc5_correct, "v2")
print("\n=== DOC5 WRONG SOURCE ===")
test_prompt(prompt_v2, doc5_wrong, "v2")

In [ ]:
def prompt_v3(n_pairs, pairs_text):
    """Discrete scoring rubric with explicit bands — forces model to pick a band."""
    return (
        f"You are a strict plagiarism detection expert.\n"
        f"Below are {n_pairs} text pair(s). Each pair shows a chunk from a SUSPICIOUS document "
        f"alongside a chunk from a CANDIDATE SOURCE document.\n\n"
        f"{pairs_text}\n\n"
        f"Your task: score the likelihood that the suspicious text was copied from this source.\n\n"
        f"SCORING RUBRIC — you MUST use one of these exact values:\n"
        f"  0.00 — No overlap. Texts share only a topic or general theme.\n"
        f"  0.25 — Weak overlap. Similar vocabulary, different sentence structure. "
        f"         OR same-author reuse (different volumes of the same work).\n"
        f"  0.50 — Moderate overlap. Some shared phrases but unclear if copied.\n"
        f"  0.85 — Synonym-swap obfuscation. Same sentence structure and narrative sequence "
        f"         with content words replaced by synonyms. Named entities still match.\n"
        f"  0.95 — Near-verbatim. Multiple pairs show verbatim or near-verbatim copying "
        f"         with unique shared phrases, names, or sequences.\n"
        f"  1.00 — Exact verbatim copy across multiple pairs.\n\n"
        f"RULES:\n"
        f"- Pick 0.85 if sentence structure is preserved with synonym substitution.\n"
        f"- Pick 0.00-0.25 if texts share a topic/era but sentence structures differ.\n"
        f"- Same-author reuse (different volumes/editions of same work) = 0.25, NOT plagiarism.\n"
        f"- You MUST return exactly one of: 0.00, 0.25, 0.50, 0.85, 0.95, 1.00.\n\n"
        f"Respond with ONLY a JSON object — no markdown, no explanation — with keys: "
        f"score (float 0-1), is_likely_source (bool), reasoning (string)."
    )

print("=== DOC5 CORRECT SOURCE ===")
test_prompt(prompt_v3, doc5_correct, "v3")
print("\n=== DOC5 WRONG SOURCE ===")
test_prompt(prompt_v3, doc5_wrong, "v3")

In [ ]:
# Critical test: doc2 same-author must score 0.25 with v3, not 0.85
doc2_sameauthor = load_pairs("part1__suspicious-document00002.txt", "part15__source-document07069.txt")
print(f"Doc2 same-author pairs: {len(doc2_sameauthor)}")

print("\n=== DOC2 SAME-AUTHOR (should score <= 0.25) ===")
test_prompt(prompt_v3, doc2_sameauthor, "v3")

## High-obfuscation test cases (word-salad / obfuscation=high)

These docs have `ret@20=1.00` (retrieval worked) but the LLM confirmed wrong sources at 0.85/0.95.

**Root cause identified from pair inspection:**
- `obfuscation=high` in PAN 2011 = machine-generated word-salad. Sentences are grammatically incoherent
  (`"they make the gusto religion"`, `"she take Braun'mho"`). Structure is destroyed, not just words swapped.
- `obfuscation=low` = synonym-swap. Sentences are still grammatically coherent — only content words replaced.
- The v3 rubric fires `0.85` (synonym-swap band) on word-salad too, because named entities survive in both.
- **Fix (v4):** Add a coherence check — if suspicious text is grammatically incoherent/word-salad,
  named entity overlap alone is insufficient for 0.85. Score 0.25 instead.

| Doc | FP source (wrong, confirmed) | Score | Correct source | Score |
|-----|------------------------------|-------|----------------|-------|
| doc10 | part18__source-document08569.txt | 0.850 | part15__source-document07440.txt | 0.950 |
| doc15 | part10__source-document04663.txt | 0.950 | part2__source-document00853.txt  | 0.850 |
| doc15 | part16__source-document07681.txt | 0.950 | — | — |

In [ ]:
# Load high-obfuscation test pairs
# Doc 10 — FP wrong source scored 0.850 with v3, correct scored 0.950
doc10_fp      = load_pairs("part1__suspicious-document00010.txt", "part18__source-document08569.txt")
doc10_correct = load_pairs("part1__suspicious-document00010.txt", "part15__source-document07440.txt")

# Doc 15 — FP wrong sources scored 0.950 with v3, correct scored 0.850
doc15_fp1     = load_pairs("part1__suspicious-document00015.txt", "part10__source-document04663.txt")
doc15_fp2     = load_pairs("part1__suspicious-document00015.txt", "part16__source-document07681.txt")
doc15_correct = load_pairs("part1__suspicious-document00015.txt", "part2__source-document00853.txt")

print(f"Doc10 FP pairs:      {len(doc10_fp)}")
print(f"Doc10 correct pairs: {len(doc10_correct)}")
print(f"Doc15 FP1 pairs:     {len(doc15_fp1)}")
print(f"Doc15 FP2 pairs:     {len(doc15_fp2)}")
print(f"Doc15 correct pairs: {len(doc15_correct)}")

## Prompt v4 — coherence guard against word-salad FP

**Change from v3:** Added a new rule distinguishing word-salad (obfuscation=high) from synonym-swap (obfuscation=low).

- Synonym-swap (obf=low): suspicious text is **grammatically coherent** — sentences parse correctly, only content words are replaced with synonyms. Structure is intact.
- Word-salad (obf=high): suspicious text is **grammatically incoherent** — sentences do not parse, words appear in wrong positions, output reads as nonsense (`"they make the gusto religion"`).

The v3 0.85 band was firing on word-salad because named entities survive both obfuscation types. v4 adds: **if the suspicious text is incoherent/ungrammatical, named entity overlap alone → 0.25, not 0.85.**

**Target scores:**
| Case | Expected |
|------|----------|
| doc5 correct (obf=low, synonym-swap, coherent) | 0.85 |
| doc5 wrong (different topic) | 0.00 |
| doc2 same-author | 0.25 |
| doc10 FP (word-salad + named entity overlap) | ≤ 0.25 |
| doc10 correct (near-verbatim) | 0.95 |
| doc15 FP1/FP2 (word-salad + topical match) | ≤ 0.25 |
| doc15 correct (word-salad but matching) | ≥ 0.85 |

In [ ]:
def prompt_v4(n_pairs, pairs_text):
    """v3 rubric + coherence guard: distinguishes word-salad (obf=high) from synonym-swap (obf=low).
    
    Key addition: if the SUSPICIOUS text is grammatically incoherent/word-salad, named entity
    overlap alone is NOT sufficient for 0.85. Must score 0.25 instead.
    Synonym-swap (obf=low) preserves grammatical coherence; word-salad (obf=high) does not.
    """
    return (
        f"You are a strict plagiarism detection expert.\n"
        f"Below are {n_pairs} text pair(s). Each pair shows a chunk from a SUSPICIOUS document "
        f"alongside a chunk from a CANDIDATE SOURCE document.\n\n"
        f"{pairs_text}\n\n"
        f"Your task: score the likelihood that the suspicious text was copied from this source.\n\n"
        f"SCORING RUBRIC — you MUST use one of these exact values:\n"
        f"  0.00 — No overlap. Texts share only a topic or general theme.\n"
        f"  0.25 — Weak overlap. Similar vocabulary or named entities but different structure.\n"
        f"         OR same-author reuse (different volumes of the same work).\n"
        f"         OR suspicious text is grammatically incoherent/word-salad (unreadable nonsense).\n"
        f"  0.50 — Moderate overlap. Some shared phrases but unclear if copied.\n"
        f"  0.85 — Synonym-swap obfuscation. Suspicious text is GRAMMATICALLY COHERENT but content\n"
        f"         words are replaced with synonyms. Sentence structure and narrative sequence are\n"
        f"         preserved. Named entities (proper nouns) still match the source.\n"
        f"  0.95 — Near-verbatim. Multiple pairs show verbatim or near-verbatim copying\n"
        f"         with unique shared phrases, names, or sequences.\n"
        f"  1.00 — Exact verbatim copy across multiple pairs.\n\n"
        f"RULES:\n"
        f"- Pick 0.85 ONLY if: (1) sentence structure is preserved AND (2) suspicious text is "
        f"grammatically readable/coherent (not word-salad).\n"
        f"- If the suspicious text reads as nonsense or word-salad (sentences do not parse, words "
        f"are in wrong grammatical positions), pick 0.25 at most — even if named entities match.\n"
        f"- Pick 0.00-0.25 if texts share a topic/era but sentence structures differ.\n"
        f"- Same-author reuse (different volumes/editions of same work) = 0.25, NOT plagiarism.\n"
        f"- You MUST return exactly one of: 0.00, 0.25, 0.50, 0.85, 0.95, 1.00.\n\n"
        f"Respond with ONLY a JSON object — no markdown, no explanation — with keys: "
        f"score (float 0-1), is_likely_source (bool), reasoning (string)."
    )

# Verify v4 still passes existing tests before running new ones
print("=== REGRESSION: DOC5 CORRECT (should be 0.85) ===")
test_prompt(prompt_v4, doc5_correct, "v4")
print("\n=== REGRESSION: DOC5 WRONG (should be 0.00) ===")
test_prompt(prompt_v4, doc5_wrong, "v4")
print("\n=== REGRESSION: DOC2 SAME-AUTHOR (should be 0.25) ===")
test_prompt(prompt_v4, doc2_sameauthor, "v4")

In [ ]:
# New tests: high-obfuscation FP cases that v3 got wrong
print("=== DOC10 FP (word-salad, should be <= 0.25) ===")
test_prompt(prompt_v4, doc10_fp, "v4")

print("\n=== DOC10 CORRECT (near-verbatim, should be 0.95) ===")
test_prompt(prompt_v4, doc10_correct, "v4")

print("\n=== DOC15 FP1 (word-salad+topical, should be <= 0.25) ===")
test_prompt(prompt_v4, doc15_fp1, "v4")

print("\n=== DOC15 FP2 (word-salad+topical, should be <= 0.25) ===")
test_prompt(prompt_v4, doc15_fp2, "v4")

print("\n=== DOC15 CORRECT (word-salad matching, should be >= 0.85) ===")
test_prompt(prompt_v4, doc15_correct, "v4")